<a href="https://colab.research.google.com/github/jsoook-dt/python-colab-analysis-study/blob/sook/project01_sook_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import kagglehub

# 1. Download the latest version of the dataset
path = kagglehub.dataset_download("manishabhatt22/marketing-campaign-performance-dataset")
print("Dataset path:", path)

# 2. Load the CSV file into a DataFrame
# Combines the download path with the specific filename
df = pd.read_csv(f"{path}/marketing_campaign_dataset.csv")

# 3. Verify data loading (Check the first 5 rows)
print("Data Sample:")
display(df.head())

# 4. Check data types and structure (Crucial for marketing analysis)
print("\nData Information:")
df.info()

100%|██████████| 5.02M/5.02M [00:00<00:00, 125MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/manishabhatt22/marketing-campaign-performance-dataset/versions/1
Data Sample:


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,"$16,174.00",6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,"$11,566.00",5.61,New York,German,116,7523,7,Fashionistas,2021-01-02
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,"$10,200.00",7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,"$12,724.00",5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,"$16,452.00",6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05



Data Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Campaign_ID       200000 non-null  int64  
 1   Company           200000 non-null  object 
 2   Campaign_Type     200000 non-null  object 
 3   Target_Audience   200000 non-null  object 
 4   Duration          200000 non-null  object 
 5   Channel_Used      200000 non-null  object 
 6   Conversion_Rate   200000 non-null  float64
 7   Acquisition_Cost  200000 non-null  object 
 8   ROI               200000 non-null  float64
 9   Location          200000 non-null  object 
 10  Language          200000 non-null  object 
 11  Clicks            200000 non-null  int64  
 12  Impressions       200000 non-null  int64  
 13  Engagement_Score  200000 non-null  int64  
 14  Customer_Segment  200000 non-null  object 
 15  Date              200000 non-null  object 
dtypes

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1) Cleaning / Type casting
# -----------------------------
df = df.copy()

# (1) Acquisition_Cost: "$16,174.00" -> 16174.00 (float)
df["Acquisition_Cost_num"] = (
    df["Acquisition_Cost"].astype(str)
      .str.replace(r"[^0-9\.\-]", "", regex=True)
)
df["Acquisition_Cost_num"] = pd.to_numeric(df["Acquisition_Cost_num"], errors="coerce")

# (2) Date: object -> datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# (3) Duration: "30 days" -> 30 (int)
df["Duration_days"] = (
    df["Duration"].astype(str)
      .str.extract(r"(\d+)", expand=False)
)
df["Duration_days"] = pd.to_numeric(df["Duration_days"], errors="coerce").astype("Int64")

# -----------------------------
# 2) KPI Engineering
# -----------------------------
# CTR = clicks / impressions
df["CTR"] = df["Clicks"] / df["Impressions"].replace(0, np.nan)

# Conversion_Rate가 0~1로 들어오는 걸 보면 보통 "클릭 대비 전환율(CVR)"로 해석하는 게 자연스러움
# Conversions(추정) = CVR * Clicks
df["Conversions_est"] = df["Conversion_Rate"] * df["Clicks"]

# CPC = spend / clicks  (여기서는 Acquisition_Cost를 spend로 간주)
df["CPC"] = df["Acquisition_Cost_num"] / df["Clicks"].replace(0, np.nan)

# CPA = spend / conversions (전환은 추정치라 CPA도 추정치)
df["CPA_est"] = df["Acquisition_Cost_num"] / df["Conversions_est"].replace(0, np.nan)

# -----------------------------
# 3) Sprint 1 요약테이블 (가중치 기반)
#   - 전체 비율형 지표는 "합으로 계산"하는 게 안전함
# -----------------------------
def summarize(group_col):
    g = df.groupby(group_col, dropna=False).agg(
        Spend=("Acquisition_Cost_num", "sum"),
        Clicks=("Clicks", "sum"),
        Impressions=("Impressions", "sum"),
        Conversions_est=("Conversions_est", "sum"),
        ROI_mean=("ROI", "mean"),
        ROI_median=("ROI", "median"),
        N=("Campaign_ID", "count")
    ).reset_index()

    g["CTR"] = g["Clicks"] / g["Impressions"].replace(0, np.nan)
    g["CVR_est"] = g["Conversions_est"] / g["Clicks"].replace(0, np.nan)
    g["CPC"] = g["Spend"] / g["Clicks"].replace(0, np.nan)
    g["CPA_est"] = g["Spend"] / g["Conversions_est"].replace(0, np.nan)

    return g

channel_summary = summarize("Channel_Used").sort_values("ROI_mean", ascending=False)
type_summary = summarize("Campaign_Type").sort_values("CVR_est", ascending=False)
duration_summary = summarize("Duration_days").sort_values("ROI_mean", ascending=False)

display(channel_summary.head(20))
display(type_summary.head(20))
display(duration_summary.head(20))

# -----------------------------
# 4) Sprint 1 질문에 바로 답하기용 Top N
# -----------------------------
# (1) ROI 최고 채널
top_roi_channel = channel_summary.iloc[0][["Channel_Used", "ROI_mean", "CTR", "CVR_est", "CPA_est", "N"]]
print("Top ROI Channel:", top_roi_channel.to_dict())

# (2) 전환율(CVR_est) 최고 캠페인 타입
top_cvr_type = type_summary.iloc[0][["Campaign_Type", "CVR_est", "ROI_mean", "CPA_est", "N"]]
print("Top CVR Campaign_Type:", top_cvr_type.to_dict())

# (3) Acquisition Cost(Spend) vs ROI 관계 (상관계수)
corr = df[["Acquisition_Cost_num", "ROI"]].corr(numeric_only=True).iloc[0,1]
print("Correlation(Spend, ROI):", corr)

,Channel_Used,Spend,Clicks,Impressions,Conversions_est,ROI_mean,ROI_median,N,CTR,CVR_est,CPC,CPA_est
1,Facebook,410595258.0,18037947,180659428,1446290.30,5.018699,5.040,32819,0.099845,0.080180,22.762860,283.895466
4,Website,416593500.0,18414628,183806353,1477710.16,5.014167,5.030,33360,0.100185,0.080247,22.622966,281.918276
2,Google Ads,418912314.0,18340807,185006879,1468753.09,5.003141,5.010,33438,0.099136,0.080081,22.840452,285.216295
0,Email,420874104.0,18493963,184801107,1485393.65,4.996487,5.000,33599,0.100075,0.080318,22.757378,283.341796
5,YouTube,416778582.0,18350407,183448082,1463568.01,4.993754,4.970,33392,0.100031,0.079757,22.712226,284.768852
3,Instagram,417124850.0,18316654,183738455,1462864.48,4.988706,4.985,33392,0.099689,0.079865,22.772983,285.142510


,Campaign_Type,Spend,Clicks,Impressions,Conversions_est,ROI_mean,ROI_median,N,CTR,CVR_est,CPC,CPA_est
2,Influencer,502400525.0,22037657,220769081,1771218.18,5.011068,5.02,40169,0.099822,0.080372,22.797366,283.646888
4,Social Media,498218100.0,21955724,219056401,1758443.91,4.991784,4.99,39817,0.100229,0.080090,22.691946,283.328969
3,Search,501911760.0,22032144,221415139,1764203.50,5.008357,5.00,40157,0.099506,0.080074,22.780886,284.497656
0,Display,500158774.0,22030979,220074756,1763752.74,5.006551,5.02,39987,0.100107,0.080058,22.702521,283.576469
1,Email,498189449.0,21897902,220144927,1746961.36,4.994295,5.00,39870,0.099470,0.079778,22.750556,285.174853


,Duration_days,Spend,Clicks,Impressions,Conversions_est,ROI_mean,ROI_median,N,CTR,CVR_est,CPC,CPA_est
1,30,627695470.0,27650863,276143827,2218662.17,5.008887,5.02,50255,0.100132,0.080238,22.700755,282.916200
3,60,624061965.0,27399624,273495631,2197184.12,5.006480,5.02,49866,0.100183,0.080190,22.776297,284.028070
2,45,626505552.0,27513257,276092560,2199162.42,4.997627,5.00,50100,0.099652,0.079931,22.771043,284.883711
0,15,622615621.0,27390662,275728286,2189570.98,4.996720,5.00,49779,0.099339,0.079939,22.730945,284.355075


Top ROI Channel: {'Channel_Used': 'Facebook', 'ROI_mean': 5.018698619702002, 'CTR': 0.09984503548854368, 'CVR_est': 0.08018042740673315, 'CPA_est': 283.8954655230696, 'N': 32819}
Top CVR Campaign_Type: {'Campaign_Type': 'Influencer', 'CVR_est': 0.08037234539043783, 'ROI_mean': 5.011068236699943, 'CPA_est': 283.64688815468236, 'N': 40169}
Correlation(Spend, ROI): 0.004584823490700722


In [4]:
## EN: Clean → Create KPIs → Aggregate summaries → Answer Sprint 1 questions.
##KR: 전처리 → KPI 생성 → 요약테이블 집계 → Sprint 1 질문에 대한 답 출력. 굵은 텍스트



## 1-1. Acquisition Cost: string → numeric

## EN: Remove $ and commas from Acquisition_Cost and convert to float.
## KR: Acquisition_Cost에서 $, , 등을 제거해 숫자(float)로 변환한다.

df["Acquisition_Cost_num"] = (
    df["Acquisition_Cost"].astype(str)
      .str.replace(r"[^0-9\.\-]", "", regex=True)
)
df["Acquisition_Cost_num"] = pd.to_numeric(df["Acquisition_Cost_num"], errors="coerce")


In [6]:
# 1-2. Date: string → datetime

## EN: Parse Date into datetime for time-based analysis.
## KR: 시계열 분석을 위해 Date를 datetime으로 변환한다.

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")